# 02. FLUX image server on a Colab GPU
The Streamlit app has no GPU, so it calls this server. **Runtime > Change runtime type > GPU.**
Colab sessions are temporary: when it stops, the app falls back to the demo poster.

Secrets needed (key icon): `SERVER_TOKEN` (make up a long random string), `HF_TOKEN` (a Hugging Face token; accept the FLUX.1-schnell terms on its model page if asked).

In [ ]:
!git clone https://github.com/<your-user>/lyric2poster-open.git app 2>/dev/null || (cd app && git pull)
%cd app
!pip -q install -r server/requirements.txt

In [ ]:
from google.colab import userdata
import os, subprocess, time
os.environ['SERVER_TOKEN'] = userdata.get('SERVER_TOKEN')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['MODEL_ID'] = 'black-forest-labs/FLUX.1-schnell'
server = subprocess.Popen(['uvicorn', 'server.image_server:app', '--port', '8000'])
time.sleep(5)
!curl -s localhost:8000/health

## Open a public tunnel
This downloads Cloudflare's `cloudflared` and prints a `https://....trycloudflare.com` address. Put it into Streamlit secrets as `L2P_REMOTE_URL` and set `L2P_REMOTE_TOKEN` to your `SERVER_TOKEN`.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'], stderr=subprocess.PIPE, text=True)
for line in tunnel.stderr:
    if 'trycloudflare.com' in line:
        print(line.strip()); break

In [ ]:
# Quick test from inside Colab (the first call loads the model and is slow)
import requests, io
from PIL import Image
r = requests.post('http://localhost:8000/generate', headers={'Authorization': 'Bearer ' + os.environ['SERVER_TOKEN']},
                  json={'prompt': 'a paper-cut collage of a harbor at dusk, no text', 'seed': 1}, timeout=600)
print(r.status_code, r.headers.get('X-Model'))
Image.open(io.BytesIO(r.content))